In [ ]:
!pip install --upgrade datasets transformers

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd

data = pd.read_excel("skincare-labeled.xlsx")

# Data sudah diberi label sentimen secara manual (0=Positive, 1=Neutral, 2=Negative)
data.head()

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_ib = 1

ib_results = []
all_y_true_ib = []
all_y_pred_ib = []

all_train_loss = []
all_eval_loss = []
all_eval_acc = []

label_map = {0:"Positive", 1:"Neutral", 2:"Negative"}

In [ ]:
import numpy as np
import torch
import random
from sklearn.utils.class_weight import compute_class_weight
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Load tokenizer dari pre-trained model IndoBERT
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2", use_fast=True)
tokenizer.add_tokens([ 'NaN' ])

In [ ]:
# Fungsi untuk proses tokenisasi teks
def tokenize_function(examples):
  # Memotong teks jika terlalu panjang (truncation) dan menambahkan padding agar seragam (maksimal 128 token)
  return tokenizer(examples["normalized_text"], padding='max_length', truncation=True, max_length=128)

In [ ]:
# Fungsi metrik evaluasi
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  predictions = np.argmax(logits, axis=-1)

  acc = accuracy_score(labels, predictions)
  precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro', zero_division=0)

  return {
      "accuracy_ib": acc,
      "precision_macro_ib": precision,
      "recall_macro_ib": recall,
      "f1_macro_ib": f1
  }

In [ ]:
class CustomTrainer(Trainer):
  def __init__(self,class_weights=None,*args,**kwargs):
    super().__init__(*args, **kwargs)
    self.class_weights = class_weights

  def compute_loss(self,model,inputs,return_outputs=False,**kwargs):
    labels = inputs.pop("labels")
    outputs = model(**inputs)
    logits = outputs.logits
    loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
    loss = loss_fct(logits.view(-1,model.config.num_labels),labels.view(-1))

    return (loss, outputs) if return_outputs else loss

In [ ]:
for train_idx, eval_idx in skf.split(data["normalized_text"],data["label"]):
  print(f"\n===== IndoBERT FOLD {fold_ib} =====")

  # Bagi data menjadi data latih (train) dan data evaluasi (eval)
  train_data = data.iloc[train_idx]
  eval_data = data.iloc[eval_idx]

  # Ambil label dari data latih untuk menghitung bobot kelas
  labels = train_data["label"]

  # Hitung bobot kelas untuk menangani masalah imbalanced data
  class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(labels), y=labels)
  class_weights = torch.tensor(class_weights, dtype=torch.float)

  # Mengubah data dari format Pandas DataFrame ke format Dataset dari Hugging Face
  train_dataset = Dataset.from_pandas(train_data[["normalized_text","label"]])
  eval_dataset = Dataset.from_pandas(eval_data[["normalized_text","label"]])

  # Menerapkan fungsi tokenisasi
  train_dataset = train_dataset.map(tokenize_function,batched=True)
  train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

  eval_dataset = eval_dataset.map(tokenize_function,batched=True)
  eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

  # Load model IndoBERT
  model = AutoModelForSequenceClassification.from_pretrained("indobenchmark/indobert-base-p2", num_labels=3)

  # Konfigurasi training
  training_args = TrainingArguments(
    output_dir=f"./result_fold_{fold_ib}",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=True,
    eval_strategy="epoch",
    logging_strategy="epoch",
    seed=42,
    data_seed=42
  )

  # Inisialisasi trainer
  trainer = CustomTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    class_weights=class_weights
  )

  # Training
  trainer.train()

  log_history = trainer.state.log_history
  train_loss = [x['loss'] for x in log_history if 'loss' in x]
  eval_loss = [x['eval_loss'] for x in log_history if 'eval_loss' in x]
  eval_acc = [x['eval_accuracy_ib'] for x in log_history if 'eval_accuracy_ib' in x]
  epochs_train = [x['epoch'] for x in log_history if 'loss' in x]
  epochs_eval = [x['epoch'] for x in log_history if 'eval_loss' in x]

  all_train_loss.append(train_loss)
  all_eval_loss.append(eval_loss)
  all_eval_acc.append(eval_acc)

  # Prediksi sekaligus  evaluasi
  pred = trainer.predict(eval_dataset)
  y_pred = np.argmax(pred.predictions,axis=1)
  y_true = pred.label_ids

  all_y_true_ib.extend(y_true)
  all_y_pred_ib.extend(y_pred)

  # Ekstraksi hasil metrik dari proses prediksi
  result = pred.metrics
  result["fold"] = fold_ib
  ib_results.append(result)

  # Hitung distribusi sentimen
  sentiment_dist = pd.Series(y_pred).value_counts().sort_index()
  sentiment_dist.index = sentiment_dist.index.map(label_map)
  sentiment_percent = sentiment_dist / sentiment_dist.sum() * 100

  sentiment_summary = pd.DataFrame({
    "Jumlah": sentiment_dist,
    "Persentase (%)": sentiment_percent.round(2)
  })

  print(f"\nDistribusi Sentimen (IndoBERT) Fold {fold_ib}")
  print(sentiment_summary)

  # Pie Chart Distribusi Sentimen
  sentiment_summary["Jumlah"].plot(kind="pie",autopct="%1.1f%%",figsize=(3,3), title=f"Distribusi Sentimen IndoBERT Fold {fold_ib}")
  plt.ylabel("")
  plt.show()

  # Classification Report
  print(f"\nClassification Report IndoBERT Fold {fold_ib}")
  print(classification_report(y_true, y_pred, target_names=["Positive", "Neutral", "Negative"]))

  # Lanjut ke fold berikutnya
  fold_ib += 1

In [ ]:
# Hitung rata-rata tiap epoch
avg_train_loss = np.mean(all_train_loss, axis=0)
avg_eval_loss = np.mean(all_eval_loss, axis=0)
avg_eval_acc = np.mean(all_eval_acc, axis=0)

epochs = range(1, len(avg_train_loss)+1)

# Average Loss Curve
plt.figure(figsize=(6,4))

plt.plot(epochs, avg_train_loss, marker='o', label='Average Train Loss')
plt.plot(epochs, avg_eval_loss, marker='o', label='Average Eval Loss')

plt.title('Average Loss Curve (5 Folds) - IndoBERT')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.show()

# Average Accuracy Curve
plt.figure(figsize=(6,4))

plt.plot(epochs, avg_eval_acc, marker='o', color='green', label='Average Eval Accuracy')

plt.title('Average Accuracy Curve (5 Folds) - IndoBERT')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Mengambil nilai F1-Score Macro dari seluruh fold
f1_indobert = [result["test_f1_macro_ib"] for result in ib_results]

print("F1-Score Macro IndoBERT per Fold:", f1_indobert)

In [ ]:
avg_ib_accuracy = np.mean(result["test_accuracy_ib"])
avg_ib_recall = np.mean(result["test_recall_macro_ib"])
avg_ib_precision = np.mean(result["test_precision_macro_ib"])
avg_ib_f1score = np.mean(result["test_f1_macro_ib"])

print(f"Average IndoBERT Accuracy: {avg_ib_accuracy:.4f}")
print(f"Average IndoBERT Recall: {avg_ib_recall:.4f}")
print(f"Average IndoBERT Precision: {avg_ib_precision:.4f}")
print(f"Average IndoBERT F1-Score: {avg_ib_f1score:.4f}")

In [ ]:
print("Classification Report Keseluruhan (Semua Fold)")

# class-wise metrics
print(classification_report(all_y_true_ib, all_y_pred_ib, target_names=["Positive", "Neutral", "Negative"]))

# Confusion Matrix
cm = confusion_matrix(all_y_true_ib, all_y_pred_ib)

# Persentase per baris (berdasarkan label asli)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Gabungkan nilai absolut + persen
labels = np.array([
    [f"{cm[i,j]}\n({cm_percent[i,j]:.2f}%)" for j in range(cm.shape[1])]
    for i in range(cm.shape[0])
])

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=labels,
    fmt="",
    cmap="Greens",
    xticklabels=["Positive", "Neutral", "Negative"],
    yticklabels=["Positive", "Neutral", "Negative"]
)

plt.title("Confusion Matrix IndoBERT")
plt.ylabel("Label Asli (True)")
plt.xlabel("Label Prediksi (Predicted)")

plt.show()